<img src='../../../common/DH_LOGO.jpeg' align='left' width=100%/>

In [1]:
# initial setup
#%run "../../../common/0_notebooks_base_setup.py"

# DHDSPaDM3A22 - Prática Guiada: Desafio Regressão Linear Múltipla - Solução

# PRÁTICA INDEPEDIENTE 

In [2]:
# Lemos os dados e definimos a data e hora como um índice.
import numpy as np
import pandas as pd

bikes = pd.read_csv('../Data/bikes.csv', 
                    index_col = 'datetime', 
                    parse_dates = True
                   )

In [3]:
# Dado que `count` é um método dos pandas, é conveniente renomear a coluna:

bikes.rename(columns = {'count' : 'total'}, 
             inplace = True
            )

##  Adicional

### Feature Engineering (Engenharia de Recursos)

Veja se você pode criar os seguintes `features`:

- **tempo**: como um único recurso numérico (de $0$ a $23$)
- **tempo**: como um recurso categórico (use $23$ variáveis dummies)
- **dia**: como um único recurso categórico (dia = $1$ das $7$h às $20$h e dia = $0$ caso contrário)

In [4]:
# `hora` como uma variável numérica
bikes['hora'] = bikes.index.hour

In [5]:
# `hora` como variável categorica

bikes_dummies = pd.get_dummies(bikes.index.hour, 
                               drop_first = True
                              )

In [6]:
bikes_dummies.head()

,1,2,3,4,5,6,7,8,9,10,...,14,15,16,17,18,19,20,21,22,23
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
bikes_dummies.index = bikes.index

In [8]:
bikes_dummies.sample(5)

,1,2,3,4,5,6,7,8,9,10,...,14,15,16,17,18,19,20,21,22,23
datetime,,,,,,,,,,,,,,,,,,,,,
2011-07-02 05:00:00,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2011-06-12 10:00:00,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2012-06-02 01:00:00,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2011-08-09 21:00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2012-09-11 18:00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


In [9]:
# día como variável categórica
a = np.array(((bikes.index.hour) >= 7) & 
             (bikes.index.hour <= 20)
            )

In [10]:
bikes['dia'] = a.astype(int)

In [11]:
bikes['dia'].sample(5)

datetime
2011-09-04 17:00:00    1
2011-01-01 14:00:00    1
2012-06-02 17:00:00    1
2012-07-18 09:00:00    1
2011-11-18 10:00:00    1
Name: dia, dtype: int64

In [12]:
bikes  = pd.concat([bikes,bikes_dummies], 
                   axis = 1
                  )

In [13]:
bikes.head()

,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,...,14,15,16,17,18,19,20,21,22,23
datetime,,,,,,,,,,,,,,,,,,,,,
2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,...,0,0,0,0,0,0,0,0,0,0
2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,...,0,0,0,0,0,0,0,0,0,0
2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,...,0,0,0,0,0,0,0,0,0,0
2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,...,0,0,0,0,0,0,0,0,0,0
2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,...,0,0,0,0,0,0,0,0,0,0


##### Em seguida, tente usar cada um dos três recursos com a função definida `train_test_rmse()` para ver qual funciona melhor!

In [14]:
from sklearn.linear_model import LinearRegression
from sklearn import metrics

In [15]:
# Definimos uma função que aceita uma lista de recursos, faz a divisão entre treino e teste,
# reservando 25% das observações para teste e retorna o teste RMSE.

from sklearn.model_selection import train_test_split

def train_test_rmse(feature_cols):
    
    X = bikes[feature_cols]
    y = bikes.total
    
     # Como estamos trabalhando com observações ordenadas no tempo, colocamos
     # `shuffle = False` para evitar vazamento de dados

    X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle = False)

    linreg = LinearRegression()
    
    linreg.fit(X_train, y_train)
    
    y_pred = linreg.predict(X_test)
    
    return np.sqrt(metrics.mean_squared_error(y_test, y_pred))

In [16]:
train_test_rmse(['temp', 'season', 'humidity'])

208.60652132566102

In [17]:
train_test_rmse(['temp', 'season', 'humidity','dia'])

177.27663881641251

In [18]:
train_test_rmse(['temp', 'season', 'humidity','hora'])

197.75102530844302

In [19]:
train_test_rmse(['temp', 
                 'season', 
                 'humidity', 
                 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23
                ]
               )

153.94217999442378

### Criar e comparar modelos com variáveis quadráticas

Com quais variáveis você testaria?

In [20]:
bikes['temp_2'] = bikes.temp ** 2

In [21]:
bikes['humidity_2'] = bikes.humidity ** 2

In [22]:
train_test_rmse(['temp', 
                 'season', 
                 'humidity', 
                 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23,
                 'temp_2', 'humidity_2'
                ]
               )

153.2818367679594